# 🏭 ModelFlow Data Factory

This notebook generates synthetic e-commerce data and seeds it into:
1. **CSV files** in `seeds/` for `dbt seed`
2. **DuckDB tables** in `data/modelflow.duckdb` under the `bronze` schema

## Dataset Overview
| Entity     | Records | Description                          |
|------------|---------|--------------------------------------|
| Customers  | 1,000   | Consumer profiles with geo attributes |
| Products   | 50      | E-commerce catalog                   |
| Orders     | 2,000   | Transactional order records          |
| Shipments  | ~1,500  | Fulfillment / logistics records      |

In [1]:
# ─────────────────────────────────────────
# Cell 1: Imports & configuration
# ─────────────────────────────────────────
import pandas as pd
import numpy as np
import duckdb
import random
import os
from pathlib import Path
from faker import Faker
from datetime import date, timedelta

# ── Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

# ── Config
NUM_CUSTOMERS = 1000
NUM_PRODUCTS  = 50
NUM_ORDERS    = 2000

# ── Paths (relative to project root)
PROJECT_ROOT  = Path().resolve()
SEEDS_DIR     = PROJECT_ROOT / 'seeds'
DATA_DIR      = PROJECT_ROOT / 'data'
DB_PATH       = DATA_DIR / 'modelflow.duckdb'

SEEDS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Seeds dir    : {SEEDS_DIR}')
print(f'DuckDB path  : {DB_PATH}')

Project root : /Users/mkarkour/ModelFlow
Seeds dir    : /Users/mkarkour/ModelFlow/seeds
DuckDB path  : /Users/mkarkour/ModelFlow/data/modelflow.duckdb


In [2]:
# ─────────────────────────────────────────
# Cell 2: Generate Customers
# ─────────────────────────────────────────
customers = pd.DataFrame([
    {
        'cust_id'    : i,
        'name'       : fake.name(),
        'email'      : fake.unique.email(),
        'country'    : fake.country(),
        'city'       : fake.city(),
        'segment'    : random.choice(['Consumer', 'Corporate', 'Home Office']),
        'created_at' : fake.date_between(start_date='-2y', end_date='-6m'),
        'updated_at' : fake.date_between(start_date='-6m', end_date='today'),
    }
    for i in range(1, NUM_CUSTOMERS + 1)
])

print(f'Customers: {len(customers):,} rows')
customers.head(3)

Customers: 1,000 rows


,cust_id,name,email,country,city,segment,created_at,updated_at
0,1,Allison Hill,donaldgarcia@example.net,Uganda,New Roberttown,Home Office,2024-05-06,2026-02-27
1,2,Kristina Baldwin,lrobinson@example.com,Sudan,Port Lindachester,Consumer,2024-11-02,2026-02-27
2,3,Gabrielle Davis,howardmaurice@example.com,Sri Lanka,Lake Stephenville,Consumer,2026-02-07,2026-02-27


In [3]:
# ─────────────────────────────────────────
# Cell 3: Generate Products
# ─────────────────────────────────────────
CATEGORY_PRICES = {
    'Electronics': (50,  500),
    'Apparel'    : (10,  150),
    'Home'       : (20,  300),
    'Toys'       : (5,   100),
    'Sports'     : (15,  250),
}

products_list = []
for i in range(1, NUM_PRODUCTS + 1):
    category = random.choice(list(CATEGORY_PRICES.keys()))
    lo, hi   = CATEGORY_PRICES[category]
    products_list.append({
        'prod_id'      : i,
        'product_name' : fake.catch_phrase(),
        'category'     : category,
        'subcategory'  : fake.word().capitalize(),
        'price'        : round(random.uniform(lo, hi), 2),
        'cost'         : round(random.uniform(lo * 0.4, hi * 0.6), 2),
        'sku'          : fake.bothify('??-####').upper(),
        'is_active'    : random.choice([True, True, True, False]),
    })

products = pd.DataFrame(products_list)
print(f'Products: {len(products):,} rows')
products.head(3)

Products: 50 rows


,prod_id,product_name,category,subcategory,price,cost,sku,is_active
0,1,Customizable static neural-net,Sports,Attorney,194.64,93.31,PF-5352,True
1,2,Ergonomic impactful analyzer,Sports,Help,104.39,136.82,HF-0791,True
2,3,Switchable solution-oriented Graphic Interface,Sports,Which,247.66,49.97,HZ-8998,True


In [4]:
# ─────────────────────────────────────────
# Cell 4: Generate Orders
# ─────────────────────────────────────────
orders_list = []
for i in range(1, NUM_ORDERS + 1):
    order_date = fake.date_between(start_date='-1y', end_date='today')
    orders_list.append({
        'order_id'         : i,
        'cust_id'          : random.randint(1, NUM_CUSTOMERS),
        'prod_id'          : random.randint(1, NUM_PRODUCTS),
        'order_date'       : order_date,
        'quantity'         : random.randint(1, 10),
        'discount_pct'     : round(random.choice([0, 0, 0, 5, 10, 15, 20]), 2),
        'status'           : random.choice(['Shipped', 'Shipped', 'Cancelled', 'Pending', 'Returned']),
        'channel'          : random.choice(['Web', 'Mobile', 'In-Store', 'Partner']),
        'payment_method'   : random.choice(['Credit Card', 'PayPal', 'Bank Transfer', 'Crypto']),
    })

orders_df = pd.DataFrame(orders_list)
print(f'Orders: {len(orders_df):,} rows')
orders_df.head(3)

Orders: 2,000 rows


,order_id,cust_id,prod_id,order_date,quantity,discount_pct,status,channel,payment_method
0,1,779,41,2025-04-17,2,5,Shipped,In-Store,Credit Card
1,2,48,21,2025-06-16,1,0,Cancelled,In-Store,Crypto
2,3,150,16,2025-07-21,9,5,Returned,Mobile,PayPal


In [5]:
# ─────────────────────────────────────────
# Cell 5: Generate Shipments
# ─────────────────────────────────────────
CARRIERS = ['DHL', 'FedEx', 'UPS', 'USPS', 'DPD']

# Only shipped & returned orders get a shipment record
shipped_orders = orders_df[
    orders_df['status'].isin(['Shipped', 'Returned'])
].copy()

shipments_list = []
for idx, (_, row) in enumerate(shipped_orders.iterrows(), start=1):
    order_date = row['order_date']
    if isinstance(order_date, str):
        order_date = date.fromisoformat(order_date)
    ship_date     = order_date + timedelta(days=random.randint(1, 5))
    delivery_date = ship_date + timedelta(days=random.randint(1, 7))
    shipments_list.append({
        'shipment_id'       : idx,
        'order_id'          : row['order_id'],
        'carrier'           : random.choice(CARRIERS),
        'tracking_number'   : fake.bothify('??########').upper(),
        'ship_date'         : ship_date,
        'delivery_date'     : delivery_date,
        'weight_kg'         : round(random.uniform(0.1, 20.0), 2),
        'shipping_cost'     : round(random.uniform(3.0, 50.0), 2),
        'status'            : 'Delivered' if row['status'] == 'Shipped' else 'Returned',
    })

shipments_df = pd.DataFrame(shipments_list)
print(f'Shipments: {len(shipments_df):,} rows')
shipments_df.head(3)

Shipments: 1,183 rows


,shipment_id,order_id,carrier,tracking_number,ship_date,delivery_date,weight_kg,shipping_cost,status
0,1,1,FedEx,IE44218299,2025-04-21,2025-04-23,1.40,23.30,Delivered
1,2,3,USPS,NY47812248,2025-07-24,2025-07-25,6.91,35.29,Returned
2,3,10,DHL,BC05670867,2025-11-16,2025-11-18,15.28,13.02,Delivered


In [6]:
# ─────────────────────────────────────────
# Cell 6: Export CSVs to seeds/
# ─────────────────────────────────────────
csv_exports = {
    'raw_customers' : customers,
    'raw_products'  : products,
    'raw_orders'    : orders_df,
    'raw_shipments' : shipments_df,
}

for name, df in csv_exports.items():
    path = SEEDS_DIR / f'{name}.csv'
    df.to_csv(path, index=False)
    print(f'✅  Exported {len(df):>6,} rows → {path.relative_to(PROJECT_ROOT)}')

✅  Exported  1,000 rows → seeds/raw_customers.csv
✅  Exported     50 rows → seeds/raw_products.csv
✅  Exported  2,000 rows → seeds/raw_orders.csv
✅  Exported  1,183 rows → seeds/raw_shipments.csv


In [7]:
# ─────────────────────────────────────────
# Cell 7: Persist to DuckDB bronze schema
# ─────────────────────────────────────────
con = duckdb.connect(str(DB_PATH))

# Create bronze schema
con.execute('CREATE SCHEMA IF NOT EXISTS bronze')

bronze_tables = {
    'bronze.raw_customers' : customers,
    'bronze.raw_products'  : products,
    'bronze.raw_orders'    : orders_df,
    'bronze.raw_shipments' : shipments_df,
}

for table_name, df in bronze_tables.items():
    con.execute(f'CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df')
    count = con.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
    print(f'✅  Created {table_name:<30} → {count:,} rows')

con.close()
print(f'\n🦆 DuckDB database ready at: {DB_PATH}')

✅  Created bronze.raw_customers           → 1,000 rows
✅  Created bronze.raw_products            → 50 rows
✅  Created bronze.raw_orders              → 2,000 rows
✅  Created bronze.raw_shipments           → 1,183 rows

🦆 DuckDB database ready at: /Users/mkarkour/ModelFlow/data/modelflow.duckdb


In [8]:
# ─────────────────────────────────────────
# Cell 8: Quick validation
# ─────────────────────────────────────────
con = duckdb.connect(str(DB_PATH), read_only=True)

print('=== Schema: bronze ===')
tables = con.execute("SELECT table_name, estimated_size FROM duckdb_tables() WHERE schema_name='bronze'").df()
display(tables)

print('\n=== Sample: bronze.raw_orders ===')
display(con.execute('SELECT * FROM bronze.raw_orders LIMIT 5').df())

print('\n=== Orders by status ===')
display(con.execute("""
    SELECT status, COUNT(*) as cnt, ROUND(SUM(quantity),0) as total_qty
    FROM bronze.raw_orders
    GROUP BY 1
    ORDER BY 2 DESC
""").df())

con.close()

=== Schema: bronze ===


,table_name,estimated_size
0,raw_customers,1000
1,raw_orders,2000
2,raw_products,50
3,raw_shipments,1183



=== Sample: bronze.raw_orders ===


,order_id,cust_id,prod_id,order_date,quantity,discount_pct,status,channel,payment_method
0,1,779,41,2025-04-17,2,5,Shipped,In-Store,Credit Card
1,2,48,21,2025-06-16,1,0,Cancelled,In-Store,Crypto
2,3,150,16,2025-07-21,9,5,Returned,Mobile,PayPal
3,4,180,6,2025-04-22,10,20,Pending,Mobile,Crypto
4,5,935,38,2026-02-25,3,0,Pending,In-Store,Crypto



=== Orders by status ===


,status,cnt,total_qty
0,Shipped,802,4481.0
1,Pending,419,2269.0
2,Cancelled,398,2197.0
3,Returned,381,2109.0
